In [ ]:
from github import Github
from github import Auth


import os
token = os.getenv("GITHUB_TOKEN")
if not token:
    raise ValueError("Set GITHUB_TOKEN in your environment before running this notebook.")
auth = Auth.Token(token)
g = Github(auth=auth)

In [56]:
owner="Mintplex-Labs"
name="anything-llm"

repo = g.get_repo(f"{owner}/{name}") 

In [57]:
pr = repo.get_pulls(state='closed', sort='updated', direction='desc')
getpr = None
for p in pr:
    if p.changed_files < 5:
        getpr = p
        break
getpr

PullRequest(title="fix: add password character validation to onboarding single-user setup", number=5037)

In [58]:
getpr.changed_files

2

In [59]:
files = getpr.get_files()
files[0]

File(sha="6a37b113cb45a35f3ab6d21be14c1653094f4431", filename="frontend/src/pages/GeneralSettings/Security/index.jsx")

In [60]:
first = files[0]
first

File(sha="6a37b113cb45a35f3ab6d21be14c1653094f4431", filename="frontend/src/pages/GeneralSettings/Security/index.jsx")

In [63]:
changed_lines = repo.compare(getpr.base.sha, getpr.head.sha).files[0].patch
changed_lines

'@@ -199,7 +199,7 @@ function MultiUserMode() {\n   );\n }\n \n-const PW_REGEX = new RegExp(/^[a-zA-Z0-9_\\-!@$%^&*();]+$/);\n+export const PW_REGEX = new RegExp(/^[a-zA-Z0-9_\\-!@$%^&*();]+$/);\n function PasswordProtection() {\n   const [saving, setSaving] = useState(false);\n   const [hasChanges, setHasChanges] = useState(false);'

In [50]:
content_merged = repo.get_contents(first.filename, ref=getpr.head.sha)
content_base = repo.get_contents(first.filename, ref=getpr.base.sha)
file_text_merged = content_merged.decoded_content.decode("utf-8")
file_text_base = content_base.decoded_content.decode("utf-8")
file_text_merged

'import { useEffect, useState } from "react";\nimport Sidebar from "@/components/SettingsSidebar";\nimport { isMobile } from "react-device-detect";\nimport showToast from "@/utils/toast";\nimport System from "@/models/system";\nimport paths from "@/utils/paths";\nimport { AUTH_TIMESTAMP, AUTH_TOKEN, AUTH_USER } from "@/utils/constants";\nimport PreLoader from "@/components/Preloader";\nimport CTAButton from "@/components/lib/CTAButton";\nimport { useTranslation } from "react-i18next";\nimport Toggle from "@/components/lib/Toggle";\nimport {\n  USERNAME_MIN_LENGTH,\n  USERNAME_MAX_LENGTH,\n  USERNAME_PATTERN,\n} from "@/utils/username";\n\nexport default function GeneralSecurity() {\n  const { t } = useTranslation();\n  return (\n    <div className="w-screen h-screen overflow-hidden bg-theme-bg-container flex">\n      <Sidebar />\n      <div\n        style={{ height: isMobile ? "100%" : "calc(100% - 32px)" }}\n        className="relative md:ml-[2px] md:mr-[16px] md:my-[16px] md:rounded-

In [51]:
file_text_base

'import { useEffect, useState } from "react";\nimport Sidebar from "@/components/SettingsSidebar";\nimport { isMobile } from "react-device-detect";\nimport showToast from "@/utils/toast";\nimport System from "@/models/system";\nimport paths from "@/utils/paths";\nimport { AUTH_TIMESTAMP, AUTH_TOKEN, AUTH_USER } from "@/utils/constants";\nimport PreLoader from "@/components/Preloader";\nimport CTAButton from "@/components/lib/CTAButton";\nimport { useTranslation } from "react-i18next";\nimport Toggle from "@/components/lib/Toggle";\nimport {\n  USERNAME_MIN_LENGTH,\n  USERNAME_MAX_LENGTH,\n  USERNAME_PATTERN,\n} from "@/utils/username";\n\nexport default function GeneralSecurity() {\n  const { t } = useTranslation();\n  return (\n    <div className="w-screen h-screen overflow-hidden bg-theme-bg-container flex">\n      <Sidebar />\n      <div\n        style={{ height: isMobile ? "100%" : "calc(100% - 32px)" }}\n        className="relative md:ml-[2px] md:mr-[16px] md:my-[16px] md:rounded-

In [2]:
from pydantic import BaseModel, Field
class ReviewComment(BaseModel):
    file_path: str = Field(
        ...,
        description="Repository-relative file path the review comment refers to.",
    )
    line_reference: str | None = Field(
        default=None,
        description="Approximate line, block, or symbol reference if available.",
    )
    severity: str = Field(
        ...,
        description="Severity of the review comment: low, medium, or high.",
    )
    category: str = Field(
        ...,
        description="Category of the review comment such as correctness, UX, maintainability, error_handling, or consistency.",
    )
    comment: str = Field(
        ...,
        description="Concrete pull request review comment written in reviewer style.",
    )
    suggestion: str | None = Field(
        default=None,
        description="Optional fix suggestion or requested change.",
    )


class GeneratedPRReview(BaseModel):
    summary: str = Field(
        ...,
        description="Short overall summary of the generated PR review.",
    )
    comments: list[ReviewComment] = Field(
        default_factory=list,
        description="List of concrete PR review comments.",
    )

In [4]:
import json
json.dumps(GeneratedPRReview.model_json_schema(), indent=2)

'{\n  "$defs": {\n    "ReviewComment": {\n      "properties": {\n        "file_path": {\n          "description": "Repository-relative file path the review comment refers to.",\n          "title": "File Path",\n          "type": "string"\n        },\n        "line_reference": {\n          "anyOf": [\n            {\n              "type": "string"\n            },\n            {\n              "type": "null"\n            }\n          ],\n          "default": null,\n          "description": "Approximate line, block, or symbol reference if available.",\n          "title": "Line Reference"\n        },\n        "severity": {\n          "description": "Severity of the review comment: low, medium, or high.",\n          "title": "Severity",\n          "type": "string"\n        },\n        "category": {\n          "description": "Category of the review comment such as correctness, UX, maintainability, error_handling, or consistency.",\n          "title": "Category",\n          "type": "string"\n 

In [30]:
base = pr.base
head = pr.head

repo.get_contents(path=first.raw_url, ref=head.ref)

GithubException: No commit found for the ref patch-1: 404 {"message": "No commit found for the ref patch-1", "documentation_url": "https://docs.github.com/v3/repos/contents/", "status": "404"}